In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
products = pd.read_csv("products.csv")
customers = pd.read_csv("customers.csv")
promotions = pd.read_csv("promotions.csv")
geography = pd.read_csv("geography.csv")
orders = pd.read_csv("orders.csv")
order_items = pd.read_csv("order_items.csv")
payments = pd.read_csv("payments.csv")
shipments = pd.read_csv("shipments.csv")
returns = pd.read_csv("returns.csv")
reviews = pd.read_csv("reviews.csv")
inventory = pd.read_csv("inventory.csv")
web_traffic = pd.read_csv("web_traffic.csv")
sales = pd.read_csv("sales.csv")

C:\Users\Admin\AppData\Local\Temp\ipykernel_672\625093302.py:6: DtypeWarning: Columns (0: promo_id_2) have mixed types. Specify dtype option on import or set low_memory=False.
  order_items = pd.read_csv("order_items.csv")


Quesion 1: Inter-order gap trung vị

In [6]:
orders['order_date'] = pd.to_datetime(orders['order_date'])

# Sort theo customer và ngày
orders_sorted = orders.sort_values(["customer_id", "order_date"])

# Tính chênh lệch giữa các đơn liên tiếp
orders_sorted["gap"] = orders_sorted.groupby("customer_id")["order_date"].diff().dt.days

# Chỉ lấy các customer có nhiều hơn 1 đơn
valid_gaps = orders_sorted.dropna(subset=["gap"])

q1_answer = valid_gaps["gap"].median()
q1_answer

np.float64(144.0)

Quesion 2: Segment có gross margin cao nhất

In [7]:
products["gross_margin"] = (products["price"] - products["cogs"]) / products["price"]

q2_answer = (
    products.groupby("segment")["gross_margin"]
    .mean()
    .sort_values(ascending=False)
)
q2_answer

segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343
Name: gross_margin, dtype: float64

Quesion 3: Lý do trả hàng nhiều nhất của Streetwear

In [8]:
streetwear_products = products[products["category"] == "Streetwear"]

merged_returns = returns.merge(
    streetwear_products[["product_id"]],
    on="product_id",
    how="inner"
)

q3_answer = merged_returns["return_reason"].value_counts()
q3_answer

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64

Quesion 4: Traffic source có bounce rate thấp nhất

In [9]:
q4_answer = (
    web_traffic.groupby("traffic_source")["bounce_rate"]
    .mean()
    .sort_values()
)
q4_answer

traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64

Quesion 5: Tỉ lệ order items có khuyến mãi

In [10]:
total_rows = len(order_items)
promo_rows = order_items["promo_id"].notna().sum()

q5_answer = promo_rows / total_rows * 100
q5_answer

np.float64(38.663493169565214)

Quesion 6: Nhóm tuổi thường mua nhiều hàng nhất

In [11]:
orders_per_customer = orders.groupby("customer_id")["order_id"].nunique()

customers_filtered = customers.dropna(subset=["age_group"])

merged_c = customers_filtered.merge(
    orders_per_customer.rename("order_count"),
    on="customer_id",
    how="left"
).fillna({"order_count": 0})

q6_answer = (
    merged_c.groupby("age_group")["order_count"]
    .mean()
    .sort_values(ascending=False)
)
q6_answer

age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
Name: order_count, dtype: float64

Quesion 7: Vùng có tổng doanh thu cao nhất

In [12]:
sales["Date"] = pd.to_datetime(sales["Date"])

orders_geo = orders.merge(geography, on="zip", how="left")

# merge orders ↔ sales theo ngày
merged_sales = orders_geo.merge(
    sales.rename(columns={"Date": "order_date"}),
    left_on="order_date",
    right_on="order_date",
    how="inner"
)

q7_answer = merged_sales.groupby("region")["Revenue"].sum().sort_values(ascending=False)
q7_answer

region
East       1.764225e+12
Central    1.098182e+12
West       9.562682e+11
Name: Revenue, dtype: float64

Quesion 8: payment_method phổ biến nhất trong đơn cancelled

In [26]:
# Lọc đơn hàng bị hủy và thống kê phương thức thanh toán trực tiếp từ bảng orders
q8_answer = orders[orders["order_status"] == "cancelled"]["payment_method"].value_counts()

print("Thống kê phương thức thanh toán của các đơn hàng bị hủy:")
print(q8_answer)

Thống kê phương thức thanh toán của các đơn hàng bị hủy:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64


Quesion 9: Size nào có tỷ lệ trả hàng cao nhất

In [14]:
# join order_items với products để biết size
oi_prod = order_items.merge(
    products[["product_id", "size"]],
    on="product_id",
    how="left"
)

# số dòng theo size
size_total = oi_prod["size"].value_counts()

# returns theo size
returns_size = returns.merge(
    products[["product_id", "size"]],
    on="product_id",
    how="left"
)["size"].value_counts()

# tính tỉ lệ
size_return_rate = (returns_size / size_total).sort_values(ascending=False)
size_return_rate

size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
Name: count, dtype: float64

Quesion 10: Installments nào có payment_value trung bình cao nhất

In [15]:
q10_answer = (
    payments.groupby("installments")["payment_value"]
    .mean()
    .sort_values(ascending=False)
)
q10_answer

installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64